In [ ]:
# ── Cell 1 — Verify GPU ───────────────────────────────
!nvidia-smi

### Steps to Get a Clean State and Reload Updated Files

To ensure you're working with the absolute latest versions of your repository files and to clear the notebook's memory, follow these steps:

1.  **Restart the Colab Runtime**: Go to `Runtime > Restart runtime` in the Colab menu. This clears all variables and loaded modules.
2.  **Run the following cells to clean up and re-prepare the environment.**
3.  **Manually re-upload `data_processed.zip`** when prompted by the `files.upload()` cell.
4.  **Run all cells from the beginning** (or `Runtime > Run all`).

#### 1. Clean up existing cloned repository and data

In [ ]:
# Remove the cloned repository directory if it exists
!rm -rf signature-verification

# Remove the data_processed.zip if it exists
!rm -f data_processed.zip

print("Clean up complete. Now proceeding to re-clone and re-setup.")

#### 2. Re-clone the repository

In [ ]:
!git clone https://github.com/nickfrostcode/signature-verification
%cd signature-verification
print("Repository re-cloned.")

#### 3. Re-install dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install pillow-heif -q
print("Dependencies re-installed.")

#### 4. Re-upload and unzip data

**Important**: You will need to manually re-upload `data_processed.zip` when prompted by the `files.upload()` cell below.

In [ ]:
from google.colab import files, drive
import os

print("--- Data Upload/Selection ---")

choice = ''
while choice not in ['U', 'D']:
    print("Do you want to (U)pload 'data_processed.zip' or load it from (D)rive? (U/D):")
    choice = input().upper()

    if choice == 'D':
        print("\nMounting Google Drive...")
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted successfully.")
        except Exception as e:
            print(f"Could not mount Google Drive: {e}. Please choose 'U' to upload.")
            choice = '' # Reset choice to force re-prompt
    elif choice != 'U':
        print("Invalid choice. Please enter 'U' or 'D'.")
        choice = '' # Reset choice to force re-prompt

if choice == 'U':
    print("\nPlease upload 'data_processed.zip' now:")
    uploaded = files.upload()
    if 'data_processed.zip' not in uploaded:
        print("Warning: 'data_processed.zip' was not found among the uploaded files. Ensure you upload the correct file.")
elif choice == 'D':
    print("\nPlease enter the full path to 'data_processed.zip' in your Google Drive.")
    print("Example: /content/drive/MyDrive/my_data/data_processed.zip")
    drive_file_path = input("Enter path: ")

    if not drive_file_path.strip():
        print("No path provided. Please restart the cell and choose 'U' or 'D'.")
    else:
        try:
            if os.path.exists(drive_file_path):
                # Copy the file to the current working directory
                !cp "{drive_file_path}" .
                print(f"Successfully copied '{drive_file_path}' from Google Drive to current directory.")
            else:
                print(f"Error: File not found at '{drive_file_path}'. Please check the path and try again.")
        except Exception as e:
            print(f"An error occurred while copying from Drive: {e}")

In [ ]:
!unzip -q data_processed.zip
!echo "Subjects found:"
!ls data_processed/ | wc -l
print("Data re-unzipped. You are now ready to run the rest of your notebook from a clean state.")

In [ ]:
import sys
sys.path.insert(0, '/content/signature-verification')

from datasets.base_dataset import get_all_subjects, split_subjects
from datasets.pair_dataset import PairDataset
from datasets.single_dataset import SingleDataset

subjects = get_all_subjects()
train_s, val_s, test_s = split_subjects(subjects)
print(f"Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

pair_ds   = PairDataset(train_s)
single_ds = SingleDataset(train_s)
print(f"Pair dataset:   {len(pair_ds):,} pairs")
print(f"Single dataset: {len(single_ds):,} images")

In [ ]:
%run training/train_baseline.py

In [ ]:
%run training/train_siamese.py

In [9]:
from google.colab import files
files.download('saved_models/baseline_cnn.pth')
files.download('saved_models/siamese.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>